# Cis4930 Project 1
## Liam Sharkey

In [ ]:
#setup
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ast


In [ ]:
#there's a lot of stuff in there i dont want
useful_cols = ['appid', 'name', 'release_date', 'price', 'dlc_count',
                'windows', 'mac', 'linux', 'metacritic_score',
               'achievements','recommendations','supported_languages',
               'full_audio_languages','developers','publishers','categories',
               'genres','score_rank','positive','negative','average_playtime_forever',
               'average_playtime_2weeks','median_playtime_forever',
               'median_playtime_2weeks','discount','peak_ccu','tags',
               'pct_pos_total','num_reviews_total','pct_pos_recent',
               'num_reviews_recent']

#had to specify python engine because c was having problems
df = pd.read_csv('data/raw/games_march2025_full.csv', 
        on_bad_lines='skip', engine='python', usecols=useful_cols)

#fill in missing data with average where available, and sentinel where not
df = df.fillna(df.mean(numeric_only=True))
string_columns = df.select_dtypes(include=['str']).columns
df[string_columns] = df[string_columns].fillna('MISSING')
pd.isnull(df).sum().sum()

#convert bools to ints
df['windows'] = df['windows'].astype(int)
df['mac'] = df['mac'].astype(int)
df['linux'] = df['linux'].astype(int)

#convert to proper lists
df["supported_languages"] = df["supported_languages"].apply(ast.literal_eval)
df["full_audio_languages"] = df['full_audio_languages'].apply(ast.literal_eval)
df['genres'] = df['genres'].apply(ast.literal_eval)

#convert release date to datetime obj
df['release_date'] = pd.to_datetime(df['release_date'])

#df.loc[df['pct_pos_total'].idxmax()]  #for ref

In [ ]:
df.describe()

## How does accessibility effect a game

Support score represents the number of different types of people
who would be able to confortably and conveniently play the game

In [ ]:
df["accessibility_score"] = (df['windows'] + df['mac'] + df['linux'] + 
    df['supported_languages'].apply(len) + df['full_audio_languages'].apply(len))

In [ ]:
filtered_df = df[df['pct_pos_total'] > 0] #a user score of 0 implies no reviews
x = filtered_df['accessibility_score']
y = filtered_df['pct_pos_total']
plt.scatter(x, y)
plt.plot(np.unique(x), np.poly1d(np.polyfit(x, y, 1))(np.unique(x)), color='red')
plt.xlabel("Accessibility Score")
plt.ylabel("Rating")
plt.title("Correlation between accessibility and rating")
plt.savefig("./figures/accessibility_scatter.png")
plt.show

## What genre is most popular

In [ ]:
df_genres = df.explode('genres')
df_genres = df_genres[df_genres['pct_pos_total'] >= 0] #-1 doesn't mean anyhting
genre_grouped = df_genres.groupby('genres')['pct_pos_total'].median().sort_values()
genre_grouped.plot(kind='bar', color='skyblue')
plt.ylabel("Percent of reviews Positive")
plt.savefig("./figures/genre_popularity_bar.png")

pt = df_genres.pivot_table(values=['pct_pos_total', 'negative', 'positive'],
                    index='genres', aggfunc='median')
pt

## Price over time

In [ ]:
price_over_time = df.groupby(df['release_date'].dt.year)['price'].mean()
price_over_time.plot()
plt.xlabel("Month")
plt.savefig("./figures/price_over_time.png")

## Price and playtime

In [ ]:
df_filterd = df[df['average_playtime_forever'] > 0]
price_playtime = df_filterd.groupby('average_playtime_forever')['price'].median()
plt.hist(price_playtime)
plt.savefig("./figures/price_playtime_hist.png")

## Correlation between all values

In [ ]:
df_numerics = df.select_dtypes(include='number')
correlation = df_numerics.corr()
plt.figure(figsize=(20,10))
sns.heatmap(correlation, annot=True, cmap='coolwarm')
plt.savefig("./figures/correlation.png")


# Summary
-There is a slight correlation between accessibility and ratings.
-Design and illustration games are most popular, though they're likely apps more than games.
-Game price has come down and stabilized in recent years, likely due to newer free to
    play models
-Surprisingly, cheaper games tend to have more playtime on them, I think this is because
    a lot of indie games have very dedicated playerbases, and AAA free to play games
    can be very addictive. (CounterStrike, Dota, Leage of Legends, Fortnite ...)
-Interestingly, older games (lower appid) have higher reviews, and are more likely to 
    support mac and linux. Just a little interesting.